In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [17]:
stock_files = {
    "NVDA": "data/NVDA.csv",
    "MSFT": "data/MSFT.csv",
    "META": "data/META.csv",
    "AMZN": "data/AMZN.csv",
    "GOOGL": "data/GOOGL.csv",
}

news_files = {
    "NVDA": "data/NVIDIA2021_2025.xlsx",
    "MSFT": "data/MSFT2021-2025.xlsx",
    "META": "data/META2021-2025.xlsx",
    "AMZN": "data/AMA2021-2025.xlsx",
    "GOOGL": "data/GOO2021-2025.xlsx",
}

In [12]:
for ticker in stock_files:
    df = pd.read_csv(stock_files[ticker])

In [14]:
stock_dfs = {}

for ticker, path in stock_files.items():
    df = pd.read_csv(path)
    df["Date"] = pd.to_datetime(df["Date"], utc=True).dt.tz_localize(None).dt.normalize()
    stock_dfs[ticker] = df
    print(ticker, df.shape)

NVDA (1254, 10)
MSFT (1254, 10)
META (1254, 10)
AMZN (1254, 10)
GOOGL (1254, 10)


In [18]:
news_dfs = {}

for ticker, path in news_files.items():
    df = pd.read_excel(path)
    df["Date"] = pd.to_datetime(df["Date"]).dt.normalize()
    df["Ticker"] = ticker
    news_dfs[ticker] = df
    print(ticker, df.shape)

NVDA (2434, 4)
MSFT (2655, 4)
META (2411, 4)
AMZN (3982, 4)
GOOGL (1624, 4)


In [19]:
def shift_weekend_to_friday(df):
    df = df.copy()
    df["Date"] = df["Date"].apply(
        lambda d: d - pd.Timedelta(days=d.weekday() - 4) if d.weekday() >= 5 else d
    )
    return df

for ticker in news_dfs:
    news_dfs[ticker] = shift_weekend_to_friday(news_dfs[ticker])

In [22]:
merged_dfs = {}

for ticker in stock_files:
    stock = stock_dfs[ticker][["Date", "Close", "Daily_Return"]].copy()
    stock = stock.sort_values("Date").reset_index(drop=True)
    
    stock["Close_tomorrow"] = stock["Close"].shift(-1)
    stock["Return_tomorrow"] = stock["Daily_Return"].shift(-1)
    stock["Direction_tomorrow"] = (stock["Return_tomorrow"] > 0).astype(int)
    
    news = news_dfs[ticker][["Date", "Title", "Summary", "Ticker"]]  # 这行加上Ticker
    
    merged = pd.merge(news, stock, on="Date", how="inner")
    merged_dfs[ticker] = merged
    print(ticker, merged.shape)

NVDA (2403, 9)
MSFT (2640, 9)
META (2388, 9)
AMZN (3952, 9)
GOOGL (1613, 9)


In [23]:
all_news = pd.concat(merged_dfs.values(), ignore_index=True)
print(all_news.shape)
all_news.head()

(12996, 9)


,Date,Title,Summary,Ticker,Close,Daily_Return,Close_tomorrow,Return_tomorrow,Direction_tomorrow
0,2021-01-06,Nvidia's $40 Billion Deal For Arm Faces U.K. M...,Nvidia Corp.'s proposed $40 billion takeover o...,NVDA,12.579124,-0.058953,13.306580,0.057830,1
1,2021-01-22,Huawei's Honor Spinoff Unveils 5G Smartphones ...,Huawei Technologies Co.'s smartphone spinoff u...,NVDA,13.674047,-0.011177,13.614962,-0.004321,0
2,2021-02-04,"EU, U.K. to Probe Nvidia's $40 Billion Arm Acq...",The European Union and U.K. are preparing to l...,NVDA,13.625930,0.009885,13.552885,-0.005361,0
3,2021-02-12,Wedbush's Ives Sees 'Arms Race' in Chip Industry,"Dan Ives, Wedbush Securities managing director...",NVDA,14.919294,-0.018983,15.287259,0.024664,1
4,2021-02-12,"Google, Microsoft, Qualcomm Protest Nvidia's A...",Some of the world's largest technology compani...,NVDA,14.919294,-0.018983,15.287259,0.024664,1


In [25]:
all_news["Summary"] = all_news["Summary"].str.replace("\n", " ").str.replace("\r", " ")
all_news.to_csv("data/news_stock_merged.csv", index=False, encoding="utf-8-sig")
print("saved")

saved


## Data Preparation

We merge Bloomberg news data with daily stock prices for five AI stocks (NVDA, MSFT, META, AMZN, GOOGL). 

For each ticker, news articles are matched to trading days by date. Weekend news (Saturday and Sunday) is shifted to the preceding Friday, so that it is paired with the next trading day's return (Monday). The target variable `Direction_tomorrow` is the sign of the next trading day's return, where 1 = up and 0 = down.

The result is saved as `news_stock_merged.csv`.

## Method 1: Loughran-McDonald Dictionary

We use the Loughran-McDonald (LM) financial sentiment dictionary to score each news article. 
Each article receives a sentiment score based on the proportion of positive and negative words 
in the title and summary. The score is defined as (positive_count - negative_count) / total_words.

In [26]:
import pysentiment2 as ps
from sklearn.metrics import accuracy_score

lm = ps.LM()

def score_text(text):
    tokens = lm.tokenize(text)
    score = lm.get_score(tokens)
    return score["Positive"] - score["Negative"]

In [27]:
all_news["text"] = all_news["Title"].fillna("") + ". " + all_news["Summary"].fillna("")
all_news["lm_score"] = all_news["text"].apply(score_text)

print(f'Positive: {(all_news["lm_score"] > 0).sum()}')
print(f'Neutral:  {(all_news["lm_score"] == 0).sum()}')
print(f'Negative: {(all_news["lm_score"] < 0).sum()}')

Positive: 1995
Neutral:  5023
Negative: 5978


In [28]:
daily_lm = all_news.groupby(["Date", "Ticker"]).agg(
    lm_score=("lm_score", "sum"),
    article_count=("lm_score", "count"),
    Direction_tomorrow=("Direction_tomorrow", "first"),
    Return_tomorrow=("Return_tomorrow", "first")
).reset_index()

print(daily_lm.shape)
daily_lm.head()

(4064, 6)


,Date,Ticker,lm_score,article_count,Direction_tomorrow,Return_tomorrow
0,2021-01-04,AMZN,-3,3,1,0.010004
1,2021-01-04,GOOGL,-3,1,1,0.008064
2,2021-01-05,AMZN,0,2,0,-0.024897
3,2021-01-05,GOOGL,-4,1,0,-0.009868
4,2021-01-06,AMZN,-1,3,1,0.007577


In [29]:
test = daily_lm[daily_lm["Date"] >= "2025-01-01"].copy()
test["pred"] = (test["lm_score"] > 0).astype(int)

for ticker in ["NVDA", "MSFT", "META", "AMZN", "GOOGL"]:
    df = test[test["Ticker"] == ticker]
    acc = accuracy_score(df["Direction_tomorrow"], df["pred"])
    baseline = df["Direction_tomorrow"].mean()
    print(f"{ticker} | Days: {len(df)} | Accuracy: {acc:.1%} | Baseline: {baseline:.1%}")

NVDA | Days: 171 | Accuracy: 45.0% | Baseline: 55.6%
MSFT | Days: 194 | Accuracy: 42.8% | Baseline: 56.2%
META | Days: 216 | Accuracy: 50.5% | Baseline: 52.3%
AMZN | Days: 211 | Accuracy: 48.8% | Baseline: 49.8%
GOOGL | Days: 194 | Accuracy: 44.3% | Baseline: 55.2%


In [30]:
portfolio = pd.concat(stock_dfs.values(), ignore_index=True)
portfolio = portfolio.groupby("Date")["Daily_Return"].mean().reset_index()
portfolio.columns = ["Date", "portfolio_return"]
portfolio["portfolio_direction"] = (portfolio["portfolio_return"].shift(-1) > 0).astype(int)
portfolio = portfolio.dropna(subset=["portfolio_direction"])

print(portfolio.shape)
portfolio.head()

(1254, 3)


,Date,portfolio_return,portfolio_direction
0,2021-01-04,NaN,1
1,2021-01-05,0.009758,0
2,2021-01-06,-0.029583,1
3,2021-01-07,0.028871,1
4,2021-01-08,0.003287,0


In [31]:
daily_port_news = all_news.groupby("Date").agg(
    lm_score=("lm_score", "sum"),
    article_count=("lm_score", "count")
).reset_index()

daily_port_news = pd.merge(daily_port_news, portfolio[["Date", "portfolio_direction"]], on="Date", how="inner")

test_port = daily_port_news[
    (daily_port_news["Date"] >= "2025-01-01") & 
    (daily_port_news["article_count"] >= 5)
].copy()

test_port["pred"] = (test_port["lm_score"] > 0).astype(int)

acc = accuracy_score(test_port["portfolio_direction"], test_port["pred"])
baseline = test_port["portfolio_direction"].mean()
print(f"Portfolio | Days: {len(test_port)} | Accuracy: {acc:.1%} | Baseline: {baseline:.1%}")

Portfolio | Days: 234 | Accuracy: 45.3% | Baseline: 53.8%
